

**Load the Bronze table into a Spark DataFrame**


In [3]:
df = spark.sql("SELECT * FROM bronze_lakehouse.dbo.bronze_scorecard_institutions LIMIT 1000")
display(df)

StatementMeta(, 36fb64b5-b05b-4b24-b140-f267b4137d84, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b3b2c69c-00a3-4aba-b9a9-9c82ccd1ae8b)

In [4]:
df.columns

StatementMeta(, 36fb64b5-b05b-4b24-b140-f267b4137d84, 6, Finished, Available, Finished, False)

['UNITID',
 'OPEID',
 'OPEID6',
 'INSTNM',
 'CITY',
 'STABBR',
 'ZIP',
 'ACCREDAGENCY',
 'INSTURL',
 'NPCURL',
 'SCH_DEG',
 'HCM2',
 'MAIN',
 'NUMBRANCH',
 'PREDDEG',
 'HIGHDEG',
 'CONTROL',
 'ST_FIPS',
 'REGION',
 'LOCALE',
 'LOCALE2',
 'LATITUDE',
 'LONGITUDE',
 'CCBASIC',
 'CCUGPROF',
 'CCSIZSET',
 'HBCU',
 'PBI',
 'ANNHI',
 'TRIBAL',
 'AANAPII',
 'HSI',
 'NANTI',
 'MENONLY',
 'WOMENONLY',
 'RELAFFIL',
 'ADM_RATE',
 'ADM_RATE_ALL',
 'SATVR25',
 'SATVR75',
 'SATMT25',
 'SATMT75',
 'SATWR25',
 'SATWR75',
 'SATVRMID',
 'SATMTMID',
 'SATWRMID',
 'ACTCM25',
 'ACTCM75',
 'ACTEN25',
 'ACTEN75',
 'ACTMT25',
 'ACTMT75',
 'ACTWR25',
 'ACTWR75',
 'ACTCMMID',
 'ACTENMID',
 'ACTMTMID',
 'ACTWRMID',
 'SAT_AVG',
 'SAT_AVG_ALL',
 'PCIP01',
 'PCIP03',
 'PCIP04',
 'PCIP05',
 'PCIP09',
 'PCIP10',
 'PCIP11',
 'PCIP12',
 'PCIP13',
 'PCIP14',
 'PCIP15',
 'PCIP16',
 'PCIP19',
 'PCIP22',
 'PCIP23',
 'PCIP24',
 'PCIP25',
 'PCIP26',
 'PCIP27',
 'PCIP29',
 'PCIP30',
 'PCIP31',
 'PCIP38',
 'PCIP39',
 'PCIP40',

**Choosing Data Columns**

In [5]:
from pyspark.sql.functions import col
cols_to_keep = [
    "UNITID", "INSTNM", "CITY", "STABBR", "REGION", "LOCALE",
    "CONTROL", "PREDDEG",
    "ADM_RATE", "SAT_AVG", "UGDS",
    "FIRST_GEN", "AGE_ENTRY",
    "UGDS_WHITE", "UGDS_BLACK", "UGDS_HISP",
    "COSTT4_A", "PCTPELL", "PCTFLOAN",
    "RET_FT4", "C100_4", "C150_4",
    "GRAD_DEBT_MDN", "RPY_3YR_RT",
    "MD_EARN_WNE_P10"
]

df_selected = df.select(*cols_to_keep)

StatementMeta(, 36fb64b5-b05b-4b24-b140-f267b4137d84, 7, Finished, Available, Finished, False)

In [6]:
display(df_selected)

StatementMeta(, 36fb64b5-b05b-4b24-b140-f267b4137d84, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ad8a5aba-e20d-4843-8907-568d3bda8aba)

**Cleaning PrivacySuppressed Values**

In [7]:
from pyspark.sql.functions import when, col

numeric_cols= ["ADM_RATE", "SAT_AVG", "UGDS",
    "FIRST_GEN", "AGE_ENTRY",
    "UGDS_WHITE", "UGDS_BLACK", "UGDS_HISP",
    "COSTT4_A", "PCTPELL", "PCTFLOAN",
    "RET_FT4", "C100_4", "C150_4",
    "GRAD_DEBT_MDN", "RPY_3YR_RT",
    "MD_EARN_WNE_P10"
]

for c in numeric_cols:
    df_selected = df_selected.withColumn(
        c, when(col(c).isin("PrivacySuppressed", "PS"), None).otherwise(col(c)).cast("double")
    )


StatementMeta(, 36fb64b5-b05b-4b24-b140-f267b4137d84, 9, Finished, Available, Finished, False)

In [8]:
display(df_selected)

StatementMeta(, 36fb64b5-b05b-4b24-b140-f267b4137d84, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, af3174bc-1873-4b89-9e35-bb4a3023d88f)

**Standardize categorical codes**


In [ ]:
df_selected = df_selected.withColumn(
    "CONTROL_DESC",
    when(col("CONTROL") == 1, "Public")
    .when(col("CONTROL") == 2, "Private Nonprofit")
    .when(col("CONTROL") == 3, "Private For-Profit")
    .otherwise("Unknown")
)

display(df_selected)

**Renaming cryptic columns to readable names**

In [10]:
column_rename_map = {
    "UNITID": "institution_id",
    "INSTNM": "institution_name",
    "CITY": "city",
    "STABBR": "state",
    "REGION": "region",
    "LOCALE": "locale_type",
    "CONTROL": "control_code",
    "CONTROL_DESC": "control_type",          
    "PREDDEG": "predominant_degree",
    "ADM_RATE": "admission_rate",
    "SAT_AVG": "avg_sat_score",
    "UGDS": "undergrad_enrollment",
    "FIRST_GEN": "pct_first_gen_students",
    "AGE_ENTRY": "avg_age_at_entry",
    "UGDS_WHITE": "pct_white_students",
    "UGDS_BLACK": "pct_black_students",
    "UGDS_HISP": "pct_hispanic_students",
    "COSTT4_A": "avg_annual_cost",
    "PCTPELL": "pct_pell_grant_recipients",
    "PCTFLOAN": "pct_federal_loan_recipients",
    "RET_FT4": "retention_rate_fulltime",
    "C100_4": "grad_rate_on_time_4yr",
    "C150_4": "grad_rate_150pct_time",
    "GRAD_DEBT_MDN": "median_grad_debt",
    "RPY_3YR_RT": "loan_repayment_rate_3yr",
    "MD_EARN_WNE_P10": "median_earnings_10yr"
}

StatementMeta(, 36fb64b5-b05b-4b24-b140-f267b4137d84, 12, Finished, Available, Finished, False)

In [11]:
for old_name, new_name in column_rename_map.items():
    if old_name in df_selected.columns:
        df_selected = df_selected.withColumnRenamed(old_name, new_name)

df_selected.printSchema()

StatementMeta(, 36fb64b5-b05b-4b24-b140-f267b4137d84, 13, Finished, Available, Finished, False)

root
 |-- institution_id: string (nullable = true)
 |-- institution_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- region: string (nullable = true)
 |-- locale_type: string (nullable = true)
 |-- control_code: string (nullable = true)
 |-- predominant_degree: string (nullable = true)
 |-- admission_rate: double (nullable = true)
 |-- avg_sat_score: double (nullable = true)
 |-- undergrad_enrollment: double (nullable = true)
 |-- pct_first_gen_students: double (nullable = true)
 |-- avg_age_at_entry: double (nullable = true)
 |-- pct_white_students: double (nullable = true)
 |-- pct_black_students: double (nullable = true)
 |-- pct_hispanic_students: double (nullable = true)
 |-- avg_annual_cost: double (nullable = true)
 |-- pct_pell_grant_recipients: double (nullable = true)
 |-- pct_federal_loan_recipients: double (nullable = true)
 |-- retention_rate_fulltime: double (nullable = true)
 |-- grad_rate_on_time_4yr: double (n

In [16]:
from pyspark.sql.functions import round as spark_round

numeric_cols_renamed = [
    "admission_rate", "avg_sat_score", "undergrad_enrollment",
    "pct_first_gen_students", "avg_age_at_entry",
    "pct_white_students", "pct_black_students", "pct_hispanic_students",
    "avg_annual_cost", "pct_pell_grant_recipients", "pct_federal_loan_recipients",
    "retention_rate_fulltime", "grad_rate_on_time_4yr", "grad_rate_150pct_time",
    "median_grad_debt", "loan_repayment_rate_3yr", "median_earnings_10yr"
]

for c in numeric_cols_renamed:
    if c in df_selected.columns:
        df_selected = df_selected.withColumn(c, spark_round(col(c), 2))

df_selected.printSchema()

StatementMeta(, 36fb64b5-b05b-4b24-b140-f267b4137d84, 18, Finished, Available, Finished, False)

root
 |-- institution_id: string (nullable = true)
 |-- institution_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- region: string (nullable = true)
 |-- locale_type: string (nullable = true)
 |-- control_code: string (nullable = true)
 |-- predominant_degree: string (nullable = true)
 |-- admission_rate: double (nullable = true)
 |-- avg_sat_score: double (nullable = true)
 |-- undergrad_enrollment: double (nullable = true)
 |-- pct_first_gen_students: double (nullable = true)
 |-- avg_age_at_entry: double (nullable = true)
 |-- pct_white_students: double (nullable = true)
 |-- pct_black_students: double (nullable = true)
 |-- pct_hispanic_students: double (nullable = true)
 |-- avg_annual_cost: double (nullable = true)
 |-- pct_pell_grant_recipients: double (nullable = true)
 |-- pct_federal_loan_recipients: double (nullable = true)
 |-- retention_rate_fulltime: double (nullable = true)
 |-- grad_rate_on_time_4yr: double (n

In [17]:
display(df_selected)

StatementMeta(, 36fb64b5-b05b-4b24-b140-f267b4137d84, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 29bc98cf-974c-4905-9b9c-1b29eea8d9af)

In [18]:
df_selected.write.format("delta").mode("overwrite").saveAsTable("silver_scorecard_institutions")

StatementMeta(, 36fb64b5-b05b-4b24-b140-f267b4137d84, 20, Finished, Available, Finished, False)

In [1]:
%%sql
SELECT COUNT(*) FROM bronze_lakehouse.dbo.bronze_scorecard_institutions;

StatementMeta(, f7655ed3-0390-4421-9841-9fe0516cf5a5, 2, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [3]:
%%sql
SELECT STABBR, COUNT(*) AS institution_count
FROM bronze_lakehouse.dbo.bronze_scorecard_institutions
GROUP BY STABBR
ORDER BY institution_count DESC;

StatementMeta(, f7655ed3-0390-4421-9841-9fe0516cf5a5, 4, Finished, Available, Finished, False)

<Spark SQL result set with 59 rows and 2 fields>

In [4]:
%%sql
SELECT COUNT(*) FROM bronze_lakehouse.dbo.silver_scorecard_institutions;

StatementMeta(, f7655ed3-0390-4421-9841-9fe0516cf5a5, 5, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [6]:
%%sql
CREATE OR REPLACE TABLE silver_scorecard_institutions AS
SELECT
    UNITID AS institution_id,
    INSTNM AS institution_name,
    CITY AS city,
    STABBR AS state
 
FROM bronze_scorecard_institutions

StatementMeta(, f7655ed3-0390-4421-9841-9fe0516cf5a5, 7, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [7]:
%%sql
SELECT COUNT(*) FROM bronze_lakehouse.dbo.silver_scorecard_institutions;

StatementMeta(, f7655ed3-0390-4421-9841-9fe0516cf5a5, 8, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [8]:
print(spark.read.table("bronze_scorecard_institutions").columns)

StatementMeta(, f7655ed3-0390-4421-9841-9fe0516cf5a5, 10, Finished, Available, Finished, False)

['UNITID', 'OPEID', 'OPEID6', 'INSTNM', 'CITY', 'STABBR', 'ZIP', 'ACCREDAGENCY', 'INSTURL', 'NPCURL', 'SCH_DEG', 'HCM2', 'MAIN', 'NUMBRANCH', 'PREDDEG', 'HIGHDEG', 'CONTROL', 'ST_FIPS', 'REGION', 'LOCALE', 'LOCALE2', 'LATITUDE', 'LONGITUDE', 'CCBASIC', 'CCUGPROF', 'CCSIZSET', 'HBCU', 'PBI', 'ANNHI', 'TRIBAL', 'AANAPII', 'HSI', 'NANTI', 'MENONLY', 'WOMENONLY', 'RELAFFIL', 'ADM_RATE', 'ADM_RATE_ALL', 'SATVR25', 'SATVR75', 'SATMT25', 'SATMT75', 'SATWR25', 'SATWR75', 'SATVRMID', 'SATMTMID', 'SATWRMID', 'ACTCM25', 'ACTCM75', 'ACTEN25', 'ACTEN75', 'ACTMT25', 'ACTMT75', 'ACTWR25', 'ACTWR75', 'ACTCMMID', 'ACTENMID', 'ACTMTMID', 'ACTWRMID', 'SAT_AVG', 'SAT_AVG_ALL', 'PCIP01', 'PCIP03', 'PCIP04', 'PCIP05', 'PCIP09', 'PCIP10', 'PCIP11', 'PCIP12', 'PCIP13', 'PCIP14', 'PCIP15', 'PCIP16', 'PCIP19', 'PCIP22', 'PCIP23', 'PCIP24', 'PCIP25', 'PCIP26', 'PCIP27', 'PCIP29', 'PCIP30', 'PCIP31', 'PCIP38', 'PCIP39', 'PCIP40', 'PCIP41', 'PCIP42', 'PCIP43', 'PCIP44', 'PCIP45', 'PCIP46', 'PCIP47', 'PCIP48', 'PCI

In [9]:
%%sql
SELECT * FROM silver_scorecard_institutions LIMIT 1;

StatementMeta(, f7655ed3-0390-4421-9841-9fe0516cf5a5, 11, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>

In [10]:
targets = ["FIRST_GEN", "AGE_ENTRY", "C100_4", "GRAD_DEBT_MDN", "MD_EARN_WNE_P10", "RPY_3YR_RT"]
cols = spark.read.table("bronze_scorecard_institutions").columns
for t in targets:
    matches = [c for c in cols if t.replace("_", "") in c.replace("_", "").upper()]
    print(t, "->", matches)

StatementMeta(, f7655ed3-0390-4421-9841-9fe0516cf5a5, 12, Finished, Available, Finished, False)

FIRST_GEN -> ['FIRSTGEN_DEATH_YR2_RT', 'FIRSTGEN_COMP_ORIG_YR2_RT', 'FIRSTGEN_COMP_4YR_TRANS_YR2_RT', 'FIRSTGEN_COMP_2YR_TRANS_YR2_RT', 'FIRSTGEN_WDRAW_ORIG_YR2_RT', 'FIRSTGEN_WDRAW_4YR_TRANS_YR2_RT', 'FIRSTGEN_WDRAW_2YR_TRANS_YR2_RT', 'FIRSTGEN_ENRL_ORIG_YR2_RT', 'FIRSTGEN_ENRL_4YR_TRANS_YR2_RT', 'FIRSTGEN_ENRL_2YR_TRANS_YR2_RT', 'FIRSTGEN_UNKN_ORIG_YR2_RT', 'FIRSTGEN_UNKN_4YR_TRANS_YR2_RT', 'FIRSTGEN_UNKN_2YR_TRANS_YR2_RT', 'FIRSTGEN_DEATH_YR3_RT', 'FIRSTGEN_COMP_ORIG_YR3_RT', 'FIRSTGEN_COMP_4YR_TRANS_YR3_RT', 'FIRSTGEN_COMP_2YR_TRANS_YR3_RT', 'FIRSTGEN_WDRAW_ORIG_YR3_RT', 'FIRSTGEN_WDRAW_4YR_TRANS_YR3_RT', 'FIRSTGEN_WDRAW_2YR_TRANS_YR3_RT', 'FIRSTGEN_ENRL_ORIG_YR3_RT', 'FIRSTGEN_ENRL_4YR_TRANS_YR3_RT', 'FIRSTGEN_ENRL_2YR_TRANS_YR3_RT', 'FIRSTGEN_UNKN_ORIG_YR3_RT', 'FIRSTGEN_UNKN_4YR_TRANS_YR3_RT', 'FIRSTGEN_UNKN_2YR_TRANS_YR3_RT', 'FIRSTGEN_DEATH_YR4_RT', 'FIRSTGEN_COMP_ORIG_YR4_RT', 'FIRSTGEN_COMP_4YR_TRANS_YR4_RT', 'FIRSTGEN_COMP_2YR_TRANS_YR4_RT', 'FIRSTGEN_WDRAW_ORIG_YR4_RT', 'F

In [11]:
%%sql
CREATE OR REPLACE TABLE silver_scorecard_institutions AS
SELECT
    UNITID AS institution_id,
    INSTNM AS institution_name,
    CITY AS city,
    STABBR AS state,
    REGION AS region,
    LOCALE AS locale_type,
    CASE CONTROL
        WHEN 1 THEN 'Public'
        WHEN 2 THEN 'Private Nonprofit'
        WHEN 3 THEN 'Private For-Profit'
        ELSE 'Unknown'
    END AS control_type,
    CASE
        WHEN NULLIF(CAST(PREDDEG AS STRING), '') IS NULL THEN NULL
        ELSE CAST(PREDDEG AS INT)
    END AS predominant_degree,
    CAST(NULLIF(ADM_RATE, 'PS') AS DOUBLE) AS admission_rate,
    CAST(NULLIF(SAT_AVG, 'PS') AS DOUBLE) AS avg_sat_score,
    CAST(NULLIF(UGDS, 'PS') AS DOUBLE) AS undergrad_enrollment,
    CAST(NULLIF(FIRST_GEN, 'PS') AS DOUBLE) AS pct_first_gen_students,
    CAST(NULLIF(AGE_ENTRY, 'PS') AS DOUBLE) AS avg_age_at_entry,
    CAST(NULLIF(UGDS_WHITE, 'PS') AS DOUBLE) AS pct_white_students,
    CAST(NULLIF(UGDS_BLACK, 'PS') AS DOUBLE) AS pct_black_students,
    CAST(NULLIF(UGDS_HISP, 'PS') AS DOUBLE) AS pct_hispanic_students,
    CAST(NULLIF(COSTT4_A, 'PS') AS DOUBLE) AS avg_annual_cost,
    CAST(NULLIF(PCTPELL, 'PS') AS DOUBLE) AS pct_pell_grant_recipients,
    CAST(NULLIF(PCTFLOAN, 'PS') AS DOUBLE) AS pct_federal_loan_recipients,
    CAST(NULLIF(RET_FT4, 'PS') AS DOUBLE) AS retention_rate_fulltime,
    CAST(NULLIF(C100_4, 'PS') AS DOUBLE) AS grad_rate_on_time_4yr,
    CAST(NULLIF(C150_4, 'PS') AS DOUBLE) AS grad_rate_150pct_time,
    CAST(NULLIF(GRAD_DEBT_MDN, 'PS') AS DOUBLE) AS median_grad_debt,
    CAST(NULLIF(RPY_3YR_RT, 'PS') AS DOUBLE) AS loan_repayment_rate_3yr,
    CAST(NULLIF(MD_EARN_WNE_P10, 'PS') AS DOUBLE) AS median_earnings_10yr
FROM bronze_scorecard_institutions

StatementMeta(, f7655ed3-0390-4421-9841-9fe0516cf5a5, 13, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [12]:
%%sql
SELECT COUNT(*) FROM silver_scorecard_institutions;

StatementMeta(, f7655ed3-0390-4421-9841-9fe0516cf5a5, 14, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [13]:
%%sql
DESCRIBE silver_scorecard_institutions;

StatementMeta(, f7655ed3-0390-4421-9841-9fe0516cf5a5, 15, Finished, Available, Finished, False)

<Spark SQL result set with 25 rows and 3 fields>